# 03. Structured State Space Models: S4 & Linear Recurrent Units (LRU)

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Albert Gu et al., 2021 (*Efficiently Modeling Long Sequences with Structured State Spaces*, ICLR 2022); Antonio Orvieto et al., 2023 (*Resurrecting Recurrent Neural Networks for Long Sequences*, ICML 2023).
- **Purpose:** Bridges continuous differential equations and deep learning, providing dual operational representations: $\mathcal{O}(L \log L)$ parallel FFT training and $\mathcal{O}(1)$ step inference without an unbounded KV cache.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [xLSTM sLSTM and mLSTM](02_xlstm_slstm_and_mlstm.ipynb) $\to$ `🔗 Downstream Syntheses:` [Mamba Selective SSM and Mamba2 SSD](04_mamba_selective_ssm_and_mamba2_ssd.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Dynamic In-Context Associative Recall ("Needle in a Haystack"):** S4/LRU state transitions are time-invariant and input-independent, preventing content-based query routing; use Mamba-2 or Softmax Attention for associative recall tasks.  
> - **Unconstrained Eigenvalues:** Unclamped eigenvalues outside the unit circle ($|\lambda| \ge 1.0$) cause immediate numerical divergence.

## Mathematical Foundations & Governing Equations

Continuous state space equation with complex diagonal transition matrix $\boldsymbol{\Lambda} = \text{diag}(\boldsymbol{\lambda})$:
$$
\lambda_j = \exp(-\nu_j + i \theta_j), \quad \nu_j > 0 \implies |\lambda_j| < 1 \quad \text{(Hurwitz Stability)}
$$

### Dual Operational Representations
1. **Linear Recurrent Mode (Inference):**
$$
\mathbf{h}_t = \boldsymbol{\lambda} \odot \mathbf{h}_{t-1} + \mathbf{B} \mathbf{u}_t, \quad \mathbf{y}_t = \text{Re}(\mathbf{C} \mathbf{h}_t) + \mathbf{D} \mathbf{u}_t
$$
2. **Convolutional Mode (Training):**
$$
\mathbf{K}_k = \text{Re}(\mathbf{C} \boldsymbol{\Lambda}^k \mathbf{B}), \quad \mathbf{y} = \mathbf{u} * \mathbf{K} + \mathbf{D} \mathbf{u}
$$

### Asymptotic Time & Memory Complexity
- **Training Time:** $\mathcal{O}(L \log L)$ via FFT convolution. **Inference Time:** $\mathcal{O}(1)$ step latency.
- **Persistent State Memory:** $\mathcal{O}(D_{\text{state}})$ bounded memory footprint (zero KV cache expansion).

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_seq = weights["x_seq"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, input sequence: {list(x_seq.shape)}")

In [ ]:
def lru_recurrent_forward(x_seq, nu, theta, b_re, b_im, c_re, c_im, d_skip):
    B, T, D_in = x_seq.shape
    D_state = nu.shape[0]
    diag_mag = torch.exp(-torch.clamp(nu, min=1e-4))
    diag_re = diag_mag * torch.cos(theta)
    diag_im = diag_mag * torch.sin(theta)

    h_re = torch.zeros(B, D_state, device=x_seq.device)
    h_im = torch.zeros(B, D_state, device=x_seq.device)
    outs = []

    for t in range(T):
        xt = x_seq[:, t, :]
        bu_re, bu_im = xt @ b_re.t(), xt @ b_im.t()
        h_re_next = diag_re * h_re - diag_im * h_im + bu_re
        h_im_next = diag_re * h_im + diag_im * h_re + bu_im
        h_re, h_im = h_re_next, h_im_next
        y_t = h_re @ c_re.t() - h_im @ c_im.t() + xt * d_skip
        outs.append(y_t.unsqueeze(1))

    return torch.cat(outs, dim=1), h_re, h_im

def s4_kernel(nu, theta, b_re, b_im, c_re, c_im, length):
    diag_mag = torch.exp(-torch.clamp(nu, min=1e-4))
    diag_re = diag_mag * torch.cos(theta)
    diag_im = diag_mag * torch.sin(theta)
    p_re = torch.ones(nu.shape[0], device=nu.device)
    p_im = torch.zeros(nu.shape[0], device=nu.device)
    kernels = []
    for _ in range(length):
        pb_re = p_re.unsqueeze(1) * b_re - p_im.unsqueeze(1) * b_im
        pb_im = p_re.unsqueeze(1) * b_im + p_im.unsqueeze(1) * b_re
        kernels.append((c_re @ pb_re - c_im @ pb_im).unsqueeze(0))
        p_re, p_im = p_re * diag_re - p_im * diag_im, p_re * diag_im + p_im * diag_re
    return torch.cat(kernels, dim=0)

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
seq_out, h_re, h_im = lru_recurrent_forward(
    x_seq, weights["nu"], weights["theta"],
    weights["b_re"], weights["b_im"], weights["c_re"], weights["c_im"], weights["d_skip"]
)
err_seq = float(torch.max(torch.abs(seq_out - weights["seq_out"])))
err_hre = float(torch.max(torch.abs(h_re - weights["final_h_re"])))
err_him = float(torch.max(torch.abs(h_im - weights["final_h_im"])))
assert err_seq < 1e-6, f"LRU sequence error: {err_seq:.2e}"
assert err_hre < 1e-6, f"LRU h_re error: {err_hre:.2e}"
assert err_him < 1e-6, f"LRU h_im error: {err_him:.2e}"

k_conv = s4_kernel(weights["nu"], weights["theta"], weights["b_re"], weights["b_im"], weights["c_re"], weights["c_im"], length=4)
err_k = float(torch.max(torch.abs(k_conv - weights["conv_kernel"])))
assert err_k < 1e-6, f"S4 Conv Kernel error: {err_k:.2e}"
print(f"Parity Verified: LRU Sequence={err_seq:.2e}, Conv Kernel={err_k:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_lru = benchmark_fn(lambda: lru_recurrent_forward(
    x_seq, weights["nu"], weights["theta"],
    weights["b_re"], weights["b_im"], weights["c_re"], weights["c_im"], weights["d_skip"]
), iters=50, name="LRU Recurrent Step")
b_conv = benchmark_fn(lambda: s4_kernel(
    weights["nu"], weights["theta"], weights["b_re"], weights["b_im"], weights["c_re"], weights["c_im"], length=16
), iters=50, name="S4 Kernel Generation")
print(format_benchmark_table([b_lru, b_conv]))

# Visualization: Complex Plane Eigenvalues on the Unit Disk & Impulse Response
fig, ax = plt.subplots(1, 3, figsize=(10, 2.6), dpi=100)
theta_grid = torch.linspace(0, 2*torch.pi, 100)
ax[0].plot(torch.cos(theta_grid).numpy(), torch.sin(theta_grid).numpy(), 'k--', alpha=0.4, label="|z|=1")
diag_mag = torch.exp(-weights["nu"])
ax[0].scatter((diag_mag * torch.cos(weights["theta"])).numpy(), (diag_mag * torch.sin(weights["theta"])).numpy(), color='crimson', s=40, zorder=5)
ax[0].set_title("Hurwitz Poles in Unit Disk")
ax[0].set_xlim(-1.2, 1.2); ax[0].set_ylim(-1.2, 1.2)

ax[1].imshow(seq_out[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[1].set_title("LRU Trajectory [T, D]")

ax[2].plot(k_conv[:, 0, 0].detach().numpy(), marker='o', color='teal')
ax[2].set_title("Impulse Kernel K_t (Channel 0)")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unconstrained Eigenvalues Outside the Unit Circle (|lambda| >= 1.0)
# If nu < 0, then |lambda| = exp(-nu) > 1. Over T=30 steps, the state diverges exponentially!
nu_flawed = torch.tensor([-0.15])  # Negative damping -> |lambda| = 1.16 > 1
nu_stable = torch.tensor([0.25])   # Positive damping -> |lambda| = 0.78 < 1
T_steps = 30

h_flawed = torch.tensor([1.0])
for _ in range(T_steps):
    h_flawed = torch.exp(-nu_flawed) * h_flawed

h_stable = torch.tensor([1.0])
for _ in range(T_steps):
    h_stable = torch.exp(-nu_stable) * h_stable

print(f"Flawed State Magnitude after {T_steps} steps: {h_flawed.item():.2e}")
print(f"Stable State Magnitude after {T_steps} steps: {h_stable.item():.2e}")
assert h_flawed.item() > 50.0, "Unstable pole must cause explosive state blowup!"
print("Demonstrated essential requirement of Hurwitz contractive stability in LRU/S4.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Dual Representation Resolves the Sequential Bottleneck:** Dual recurrent and convolutional modes combine fast $\mathcal{O}(L \log L)$ parallel training with constant-memory $\mathcal{O}(1)$ auto-regressive decode.
2. **Hurwitz Stability is Guaranteed via Polar Form:** Parameterizing diagonal poles as $\lambda_j = \exp(-\nu_j + i\theta_j)$ with $\nu_j > 0$ strictly confines eigenvalues inside the unit circle.
3. **Eliminates KV Cache Footprint:** State memory remains constant regardless of sequence length, enabling 100k+ context windows on commodity hardware.

---
### Cross-Domain Generalisation Matrix
- **Seismic Waveform & Geophysical Modeling:** Captures high-frequency acoustic wave reflections across deep tectonic layers over $10^5$ timesteps.
- **Audio Synthesizer Modeling:** Discrete state space equations simulate resonant analog IIR/FIR filter dynamics with precise harmonic reproduction.
- **Continuous Medical Biosignals (ECG/EEG):** Models irregular sampling intervals by directly modulating the discretization step size $\Delta$.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_structured_state_space_models_s4_and_lru.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.md)
- **PyTorch 2.x Module:** [03_structured_state_space_models_s4_and_lru.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.py)
- **Functional JAX Module:** [03_structured_state_space_models_s4_and_lru.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.py)
- **Native Rust Candle Track:** [03_structured_state_space_models_s4_and_lru.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.rs)
- **Hardware WGSL Kernel:** [03_structured_state_space_models_s4_and_lru.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.rs)
- **Universal Golden SafeTensors:** [03_structured_state_space_models_s4_and_lru.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/03_structured_state_space_models_s4_and_lru.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 02. xLSTM sLSTM and mLSTM](02_xlstm_slstm_and_mlstm.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 04. Mamba Selective SSM and Mamba2 SSD](04_mamba_selective_ssm_and_mamba2_ssd.ipynb) &rarr;